In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import gc

In [2]:
print("="*70)
print("PHASE 3: HYBRID RECOMMENDATION - 500K REVIEW SUBSET")
print("="*70)

# ============================================================
# STEP 1: LOAD AND SAMPLE TO 500K
# ============================================================
print("\n[1/7] Loading data and sampling to 500K reviews...")

df_train_full = pd.read_csv('train_data.csv')
print(f"  Full dataset: {len(df_train_full):,} reviews")

# Sample 500K reviews
sample_size = min(500_000, len(df_train_full))
df_train = df_train_full.sample(n=sample_size, random_state=42)
print(f"✓ Sampled: {len(df_train):,} reviews")
print(f"  Items: {df_train['item_id'].nunique():,}")
print(f"  Users: {df_train['user_id'].nunique():,}")

# Load and filter item features
item_features_full = pd.read_csv('item_features.csv')
print(f"✓ Item features: {len(item_features_full):,} items")

items_in_sample = set(df_train['item_id'].unique())
item_features = item_features_full[item_features_full['item_id'].isin(items_in_sample)].reset_index(drop=True)
print(f"✓ Filtered to: {len(item_features):,} items in sample")

# Load embeddings
doc_embeddings_full = np.load('doc_embeddings.npy')
with open('w2v_model.pkl', 'rb') as f:
    w2v_model = pickle.load(f)

print(f"✓ Loaded embeddings: {doc_embeddings_full.shape}")

PHASE 3: HYBRID RECOMMENDATION - 500K REVIEW SUBSET

[1/7] Loading data and sampling to 500K reviews...
  Full dataset: 1,918,055 reviews
✓ Sampled: 500,000 reviews
  Items: 110,226
  Users: 313,340
✓ Item features: 199,368 items
✓ Filtered to: 110,226 items in sample
✓ Loaded embeddings: (1918055, 100)


In [3]:
# ============================================================
# STEP 2: COLLABORATIVE FILTERING (SVD)
# ============================================================
print("\n[2/7] Training Collaborative Filtering (SVD)...")

user_to_idx = {uid: i for i, uid in enumerate(df_train['user_id'].unique())}
item_to_idx = {iid: i for i, iid in enumerate(df_train['item_id'].unique())}

print(f"  User-Item Matrix: ({len(user_to_idx):,}, {len(item_to_idx):,})")

row_indices = df_train['user_id'].map(user_to_idx).values
col_indices = df_train['item_id'].map(item_to_idx).values
ratings = df_train['rating'].values

user_item_matrix = csr_matrix(
    (ratings, (row_indices, col_indices)),
    shape=(len(user_to_idx), len(item_to_idx))
)

print(f"  Sparsity: {(1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1])) * 100:.2f}%")

global_mean = ratings.mean()
print(f"  Global mean rating: {global_mean:.2f}")

print("  Computing biases...")
user_bias = (df_train.groupby('user_id')['rating'].mean() - global_mean).to_dict()
item_bias = (df_train.groupby('item_id')['rating'].mean() - global_mean).to_dict()

for uid in user_to_idx.keys():
    if uid not in user_bias:
        user_bias[uid] = 0.0
for iid in item_to_idx.keys():
    if iid not in item_bias:
        item_bias[iid] = 0.0

print("  Training SVD (n_components=50)...")
n_factors = 50
svd_model = TruncatedSVD(n_components=n_factors, random_state=42, n_iter=10)
user_factors = svd_model.fit_transform(user_item_matrix)
item_factors = svd_model.components_.T

print(f"✓ SVD trained")
print(f"  Explained variance: {svd_model.explained_variance_ratio_.sum():.4f}")
print(f"  User factors: {user_factors.shape}")
print(f"  Item factors: {item_factors.shape}")




[2/7] Training Collaborative Filtering (SVD)...
  User-Item Matrix: (313,340, 110,226)
  Sparsity: 100.00%
  Global mean rating: 4.23
  Computing biases...
  Training SVD (n_components=50)...
✓ SVD trained
  Explained variance: 0.0623
  User factors: (313340, 50)
  Item factors: (110226, 50)


In [4]:
# ============================================================
# STEP 3: CONTENT-BASED FEATURES
# ============================================================
print("\n[3/7] Building content-based features...")

scaler = StandardScaler()

# Sentiment
sentiment_features = item_features[['sentiment_polarity', 'sentiment_subjectivity']].fillna(0).values
sentiment_norm = scaler.fit_transform(sentiment_features)

# Topics
topic_features = item_features[[f'topic_{i}' for i in range(15)]].fillna(0).values

# Aspects
aspect_cols = ['plot', 'characters', 'writing_style', 'setting', 'emotion', 'pacing']
aspect_features = item_features[[f'aspect_{asp}' for asp in aspect_cols]].fillna(0).values

# Embeddings (100 dims)
embedding_cols = [f'embedding_{i}' for i in range(100)]
embedding_features = item_features[embedding_cols].fillna(0).astype(np.float32).values

# Combine weighted
combined_content = np.hstack([
    sentiment_norm * 0.1,
    topic_features * 0.2,
    aspect_features * 0.3,
    embedding_features * 0.4
]).astype(np.float32)

content_scaler = StandardScaler()
content_features_norm = content_scaler.fit_transform(combined_content)
content_features_norm = normalize(content_features_norm, norm='l2').astype(np.float32)

print(f"✓ Content features: {content_features_norm.shape}")
print(f"  Memory: ~{content_features_norm.nbytes / (1024**2):.1f} MB")

del sentiment_features, topic_features, aspect_features, embedding_features, combined_content
gc.collect()



[3/7] Building content-based features...
✓ Content features: (110226, 123)
  Memory: ~51.7 MB


0

In [5]:
# ============================================================
# STEP 4: COMPUTE ITEM SIMILARITY (OPTIMIZED FOR MEMORY)
# ============================================================
print("\n[4/7] Computing item similarity...")

batch_size = 1000
n_items = len(item_features)
item_similarity_dict = {}

# CRITICAL SETTINGS - Tuned to prevent memory explosion
USE_TOP_K = 1000  # Keep top 1000 most similar items (good balance)
SIMILARITY_THRESHOLD = 0.2  # Higher threshold = fewer similarities stored

print(f"  Processing {n_items:,} items in batches of {batch_size}...")
print(f"  ⚙️ Threshold: {SIMILARITY_THRESHOLD}, Top-K: {USE_TOP_K}")
print(f"  ⚠️ These settings prevent the 21GB memory explosion!")

for i in range(0, n_items, batch_size):
    batch_end = min(i + batch_size, n_items)
    batch_features = content_features_norm[i:batch_end]
    
    # Compute similarity with all items
    batch_similarity = cosine_similarity(batch_features, content_features_norm)
    
    # Store similarities with aggressive filtering
    for local_idx in range(batch_similarity.shape[0]):
        global_idx = i + local_idx
        sims = batch_similarity[local_idx]
        
        # Filter by threshold FIRST
        valid_mask = sims > SIMILARITY_THRESHOLD
        valid_indices = np.where(valid_mask)[0]
        valid_sims = sims[valid_indices]
        
        if len(valid_sims) > 0:
            # Keep only top K most similar (CRITICAL for memory!)
            if len(valid_sims) > USE_TOP_K:
                top_k_indices = np.argsort(valid_sims)[-USE_TOP_K:]
                valid_indices = valid_indices[top_k_indices]
                valid_sims = valid_sims[top_k_indices]
            
            # Sort by similarity (descending)
            sort_order = np.argsort(valid_sims)[::-1]
            
            item_similarity_dict[global_idx] = {
                'indices': valid_indices[sort_order].astype(np.int32),
                'similarities': valid_sims[sort_order].astype(np.float32)
            }
    
    if (i + batch_size) % 5000 == 0 or batch_end == n_items:
        current_avg = np.mean([len(v['indices']) for v in item_similarity_dict.values()]) if item_similarity_dict else 0
        print(f"  ✓ Processed {batch_end:,}/{n_items:,} items (avg sims: {current_avg:.0f})")
    
    # Clean up after each batch
    del batch_similarity
    gc.collect()

print(f"✓ Similarity index created")
avg_sims = np.mean([len(v['indices']) for v in item_similarity_dict.values()])
total_mem = (sum(len(v['indices']) for v in item_similarity_dict.values()) * 8) / (1024**2)
print(f"  📊 Avg similarities per item: {avg_sims:.0f}")
print(f"  💾 Total memory: ~{total_mem:.1f} MB")

# Safety check
if total_mem > 2000:
    print(f"  ⚠️ WARNING: Memory usage is high ({total_mem:.0f} MB)")
    print(f"  Consider increasing SIMILARITY_THRESHOLD or decreasing USE_TOP_K")
elif total_mem < 100:
    print(f"  ⚠️ WARNING: Memory usage is very low ({total_mem:.0f} MB)")
    print(f"  Consider decreasing SIMILARITY_THRESHOLD or increasing USE_TOP_K")
else:
    print(f"  ✅ Memory usage is optimal!")

del content_features_norm
gc.collect()



[4/7] Computing item similarity...
  Processing 110,226 items in batches of 1000...
  ⚙️ Threshold: 0.2, Top-K: 1000
  ⚠️ These settings prevent the 21GB memory explosion!
  ✓ Processed 5,000/110,226 items (avg sims: 1000)
  ✓ Processed 10,000/110,226 items (avg sims: 1000)
  ✓ Processed 15,000/110,226 items (avg sims: 1000)
  ✓ Processed 20,000/110,226 items (avg sims: 1000)
  ✓ Processed 25,000/110,226 items (avg sims: 1000)
  ✓ Processed 30,000/110,226 items (avg sims: 1000)
  ✓ Processed 35,000/110,226 items (avg sims: 1000)
  ✓ Processed 40,000/110,226 items (avg sims: 1000)
  ✓ Processed 45,000/110,226 items (avg sims: 1000)
  ✓ Processed 50,000/110,226 items (avg sims: 1000)
  ✓ Processed 55,000/110,226 items (avg sims: 1000)
  ✓ Processed 60,000/110,226 items (avg sims: 1000)
  ✓ Processed 65,000/110,226 items (avg sims: 1000)
  ✓ Processed 70,000/110,226 items (avg sims: 1000)
  ✓ Processed 75,000/110,226 items (avg sims: 1000)
  ✓ Processed 80,000/110,226 items (avg sims: 10

0

In [6]:
# ============================================================
# CHECKPOINT: SAVE INTERMEDIATE RESULTS (STEPS 2-4)
# ============================================================
print("\n[4.5/7] Saving intermediate results (checkpointing)...")

joblib.dump(user_factors, 'user_factors_500k.pkl', compress=3)
print("  ✓ user_factors_500k.pkl")

joblib.dump(item_factors, 'item_factors_500k.pkl', compress=3)
print("  ✓ item_factors_500k.pkl")

joblib.dump(user_to_idx, 'user_to_idx_500k.pkl', compress=3)
print("  ✓ user_to_idx_500k.pkl")

joblib.dump(item_to_idx, 'item_to_idx_500k.pkl', compress=3)
print("  ✓ item_to_idx_500k.pkl")

joblib.dump(user_bias, 'user_bias_500k.pkl', compress=3)
print("  ✓ user_bias_500k.pkl")

joblib.dump(item_bias, 'item_bias_500k.pkl', compress=3)
print("  ✓ item_bias_500k.pkl")

joblib.dump(item_similarity_dict, 'item_similarity_dict_500k.pkl', compress=3)
print("  ✓ item_similarity_dict_500k.pkl")

joblib.dump({'global_mean': global_mean}, 'global_stats_500k.pkl', compress=3)
print("  ✓ global_stats_500k.pkl")

print("✓ Checkpoint saved! Can resume from here if needed.")



[4.5/7] Saving intermediate results (checkpointing)...
  ✓ user_factors_500k.pkl
  ✓ item_factors_500k.pkl
  ✓ user_to_idx_500k.pkl
  ✓ item_to_idx_500k.pkl
  ✓ user_bias_500k.pkl
  ✓ item_bias_500k.pkl
  ✓ item_similarity_dict_500k.pkl
  ✓ global_stats_500k.pkl
✓ Checkpoint saved! Can resume from here if needed.


In [7]:
# ============================================================
# STEP 5: HYBRID RECOMMENDER CLASS
# ============================================================
print("\n[5/7] Building hybrid recommender...")

class HybridRecommender:
    def __init__(self, user_factors, item_factors, user_to_idx, item_to_idx,
                 item_similarity_dict, item_features, df_train,
                 user_bias, item_bias, global_mean, alpha=0.5):
        
        self.user_factors = user_factors
        self.item_factors = item_factors
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.item_similarity_dict = item_similarity_dict
        self.item_features = item_features
        self.user_bias = user_bias
        self.item_bias = item_bias
        self.global_mean = global_mean
        self.alpha = alpha
        
        self.idx_to_item = {v: k for k, v in item_to_idx.items()}
        self.user_items = df_train.groupby('user_id')['item_id'].apply(set).to_dict()
        
        self.user_item_ratings = {}
        for uid in self.user_to_idx.keys():
            self.user_item_ratings[uid] = {}
        
        for _, row in df_train.iterrows():
            uid = row['user_id']
            iid = row['item_id']
            if iid in item_to_idx:
                self.user_item_ratings[uid][self.item_to_idx[iid]] = row['rating']
    
    def get_cf_score(self, user_id, item_id):
        """CF score with bias adjustment"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        u_idx = self.user_to_idx[user_id]
        i_idx = self.item_to_idx[item_id]
        
        dot_product = np.dot(self.user_factors[u_idx], self.item_factors[i_idx])
        prediction = (self.global_mean + 
                     self.user_bias.get(user_id, 0.0) + 
                     self.item_bias.get(item_id, 0.0) + 
                     dot_product)
        
        return np.clip(prediction, 1.0, 5.0)
    
    def get_content_score(self, user_id, item_id):
        """Content-based score using similarities"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        item_idx = self.item_to_idx[item_id]
        user_rated_items = self.user_items.get(user_id, set())
        
        if not user_rated_items or item_idx not in self.item_similarity_dict:
            return self.global_mean
        
        user_item_indices = {self.item_to_idx[iid]: iid 
                            for iid in user_rated_items 
                            if iid in self.item_to_idx}
        
        if not user_item_indices:
            return self.global_mean
        
        similar_data = self.item_similarity_dict[item_idx]
        similar_indices = similar_data['indices']
        similar_sims = similar_data['similarities'].astype(np.float32)
        
        weighted_sum = 0.0
        sim_sum = 0.0
        
        for sim_idx, sim_val in zip(similar_indices, similar_sims):
            if sim_idx in user_item_indices:
                rating = self.user_item_ratings[user_id].get(sim_idx, self.global_mean)
                weighted_sum += float(sim_val) * rating
                sim_sum += float(sim_val)
        
        if sim_sum == 0:
            return self.global_mean
        
        prediction = weighted_sum / sim_sum
        return np.clip(prediction, 1.0, 5.0)

recommender = HybridRecommender(
    user_factors, item_factors, user_to_idx, item_to_idx,
    item_similarity_dict, item_features, df_train,
    user_bias, item_bias, global_mean, alpha=0.5
)
print("✓ Recommender initialized (α=0.5)")



[5/7] Building hybrid recommender...
✓ Recommender initialized (α=0.5)


In [8]:
# ============================================================
# STEP 6: EVALUATION
# ============================================================
print("\n[6/7] Evaluating performance...")

sample_size = min(5000, len(df_train))  # Reduced from 10k
test_sample = df_train.sample(n=sample_size, random_state=42)

cf_preds = []
content_preds = []
hybrid_preds = []
actual_ratings = []

print(f"Scoring {sample_size:,} predictions...")
for idx, (_, row) in enumerate(test_sample.iterrows()):
    if idx % 1000 == 0:
        print(f"  ✓ {idx:,}/{sample_size:,}")
    
    cf = recommender.get_cf_score(row['user_id'], row['item_id'])
    content = recommender.get_content_score(row['user_id'], row['item_id'])
    hybrid = 0.5 * cf + 0.5 * content
    
    cf_preds.append(cf)
    content_preds.append(content)
    hybrid_preds.append(hybrid)
    actual_ratings.append(row['rating'])

cf_rmse = np.sqrt(mean_squared_error(actual_ratings, cf_preds))
content_rmse = np.sqrt(mean_squared_error(actual_ratings, content_preds))
hybrid_rmse = np.sqrt(mean_squared_error(actual_ratings, hybrid_preds))

print(f"\n✓ Performance Metrics ({sample_size:,} samples):")
print(f"  CF Only:      RMSE = {cf_rmse:.4f}")
print(f"  Content Only: RMSE = {content_rmse:.4f}")
print(f"  Hybrid (50%): RMSE = {hybrid_rmse:.4f}")

print(f"\n  Score distributions:")
print(f"    CF:       mean={np.mean(cf_preds):.2f}, std={np.std(cf_preds):.2f}")
print(f"    Content:  mean={np.mean(content_preds):.2f}, std={np.std(content_preds):.2f}")
print(f"    Hybrid:   mean={np.mean(hybrid_preds):.2f}, std={np.std(hybrid_preds):.2f}")
print(f"    Actual:   mean={np.mean(actual_ratings):.2f}, std={np.std(actual_ratings):.2f}")




[6/7] Evaluating performance...
Scoring 5,000 predictions...
  ✓ 0/5,000
  ✓ 1,000/5,000
  ✓ 2,000/5,000
  ✓ 3,000/5,000
  ✓ 4,000/5,000

✓ Performance Metrics (5,000 samples):
  CF Only:      RMSE = 0.5762
  Content Only: RMSE = 0.1171
  Hybrid (50%): RMSE = 0.3044

  Score distributions:
    CF:       mean=4.15, std=1.20
    Content:  mean=4.23, std=1.17
    Hybrid:   mean=4.19, std=1.15
    Actual:   mean=4.23, std=1.18


In [9]:
# ============================================================
# STEP 7: SAVE MODELS
# ============================================================
print("\n[7/7] Saving models...")

joblib.dump(user_factors, 'user_factors_500k.pkl', compress=3)
joblib.dump(item_factors, 'item_factors_500k.pkl', compress=3)
joblib.dump(user_to_idx, 'user_to_idx_500k.pkl', compress=3)
joblib.dump(item_to_idx, 'item_to_idx_500k.pkl', compress=3)
joblib.dump(user_bias, 'user_bias_500k.pkl', compress=3)
joblib.dump(item_bias, 'item_bias_500k.pkl', compress=3)
joblib.dump(recommender, 'hybrid_recommender_500k.pkl', compress=3)
joblib.dump(item_similarity_dict, 'item_similarity_dict_500k.pkl', compress=3)

print("✓ All models saved (with _500k suffix)")

print("\n" + "="*70)
print("PHASE 3 COMPLETE!")
print("="*70)
print(f"\n✅ Used 500K review subset (from full {len(df_train_full):,})")
print(f"   Memory optimizations:")
print(f"   • Smaller batch size (1000 vs 2000)")
print(f"   • Higher similarity threshold (0.15 vs 0.1)")
print(f"   • Top-K filtering (500 most similar items)")
print(f"   • Aggressive garbage collection")
print(f"   • Smaller evaluation sample (5K vs 10K)")


[7/7] Saving models...
✓ All models saved (with _500k suffix)

PHASE 3 COMPLETE!

✅ Used 500K review subset (from full 1,918,055)
   Memory optimizations:
   • Smaller batch size (1000 vs 2000)
   • Higher similarity threshold (0.15 vs 0.1)
   • Top-K filtering (500 most similar items)
   • Aggressive garbage collection
   • Smaller evaluation sample (5K vs 10K)


In [13]:
"""
Improved Hybrid Recommender with:
1. Configurable alpha at prediction time
2. Variance scaling for content predictions
3. Confidence-weighted hybrid (adaptive alpha)
4. Better cold-start handling
"""

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("IMPROVED HYBRID RECOMMENDER - EVALUATION")
print("="*70)

# ============================================================
# LOAD MODEL AND DATA
# ============================================================
print("\n[1/4] Loading model and validation data...")

recommender = joblib.load('hybrid_recommender_500k.pkl')
user_to_idx = joblib.load('user_to_idx_500k.pkl')
item_to_idx = joblib.load('item_to_idx_500k.pkl')

df_val = pd.read_csv('val_data.csv')
df_val_filtered = df_val[
    (df_val['user_id'].isin(user_to_idx.keys())) & 
    (df_val['item_id'].isin(item_to_idx.keys()))
].copy()

sample_size = min(10000, len(df_val_filtered))
test_sample = df_val_filtered.sample(n=sample_size, random_state=42)

print(f"✓ Loaded {sample_size:,} validation samples")

# Compute user activity (for confidence weighting)
user_activity = {}
for uid in user_to_idx.keys():
    user_activity[uid] = len(recommender.user_item_ratings.get(uid, {}))

# ============================================================
# IMPROVEMENT 1: VARIANCE-SCALED CONTENT PREDICTIONS
# ============================================================
print("\n[2/4] Testing variance-scaled content predictions...")

def get_scaled_content_score(recommender, user_id, item_id, scale_factor=2.0):
    """
    Scale content predictions to match actual rating variance
    Problem: Content predicts too conservatively (std=0.30)
    Solution: Expand predictions around mean
    """
    prediction = recommender.get_content_score(user_id, item_id)
    global_mean = recommender.global_mean
    
    # Scale deviation from mean
    scaled = (prediction - global_mean) * scale_factor + global_mean
    return np.clip(scaled, 1.0, 5.0)

# Test different scale factors
scale_factors = [1.0, 1.5, 2.0, 2.5, 3.0]
print("\nTesting content variance scaling factors:")
print("Factor | Content RMSE | Hybrid RMSE (α=0.6)")
print("-------|--------------|--------------------")

best_scale = 1.0
best_rmse = float('inf')

for scale in scale_factors:
    content_preds = []
    for _, row in test_sample.iterrows():
        pred = get_scaled_content_score(recommender, row['user_id'], row['item_id'], scale)
        content_preds.append(pred)
    
    actual = test_sample['rating'].tolist()
    content_rmse = np.sqrt(mean_squared_error(actual, content_preds))
    
    # Test in hybrid (α=0.6)
    cf_preds = [recommender.get_cf_score(row['user_id'], row['item_id']) 
                for _, row in test_sample.iterrows()]
    hybrid_preds = [0.6*cf + 0.4*cont for cf, cont in zip(cf_preds, content_preds)]
    hybrid_rmse = np.sqrt(mean_squared_error(actual, hybrid_preds))
    
    print(f"  {scale:.1f}  |    {content_rmse:.4f}    |      {hybrid_rmse:.4f}")
    
    if hybrid_rmse < best_rmse:
        best_rmse = hybrid_rmse
        best_scale = scale

print(f"\n✓ Best scale factor: {best_scale} (Hybrid RMSE: {best_rmse:.4f})")

# ============================================================
# IMPROVEMENT 2: CONFIDENCE-WEIGHTED HYBRID
# ============================================================
print("\n[3/4] Testing confidence-weighted hybrid (adaptive alpha)...")

def get_adaptive_alpha(user_id, user_activity, min_alpha=0.3, max_alpha=0.8):
    """
    Adaptive alpha based on user activity:
    - Sparse users (few ratings): Lower alpha → More content weight
    - Active users (many ratings): Higher alpha → More CF weight
    """
    activity = user_activity.get(user_id, 0)
    
    # Sigmoid function for smooth transition
    # activity=5 → α≈0.3, activity=50 → α≈0.8
    if activity <= 5:
        return min_alpha
    elif activity >= 50:
        return max_alpha
    else:
        # Linear interpolation
        return min_alpha + (max_alpha - min_alpha) * (activity - 5) / 45

print("\nAdaptive alpha strategy:")
print("  User Activity | Alpha | Method Weighting")
print("  --------------|-------|------------------")
print("  0-5 ratings   | 0.30  | 30% CF + 70% Content (favor content)")
print("  25 ratings    | 0.55  | 55% CF + 45% Content (balanced)")
print("  50+ ratings   | 0.80  | 80% CF + 20% Content (favor CF)")

# Evaluate adaptive hybrid
adaptive_preds = []
cf_preds = []
content_preds_scaled = []

for _, row in test_sample.iterrows():
    cf = recommender.get_cf_score(row['user_id'], row['item_id'])
    content = get_scaled_content_score(recommender, row['user_id'], row['item_id'], best_scale)
    alpha = get_adaptive_alpha(row['user_id'], user_activity)
    
    adaptive = alpha * cf + (1 - alpha) * content
    
    cf_preds.append(cf)
    content_preds_scaled.append(content)
    adaptive_preds.append(adaptive)

actual = test_sample['rating'].tolist()

cf_rmse = np.sqrt(mean_squared_error(actual, cf_preds))
content_rmse = np.sqrt(mean_squared_error(actual, content_preds_scaled))
adaptive_rmse = np.sqrt(mean_squared_error(actual, adaptive_preds))

print(f"\n✓ Adaptive hybrid RMSE: {adaptive_rmse:.4f}")

# ============================================================
# IMPROVEMENT 3: COMPREHENSIVE COMPARISON
# ============================================================
print("\n[4/4] Final comparison of all methods...")

# Fixed alpha methods
alpha_06_preds = [0.6*cf + 0.4*cont for cf, cont in zip(cf_preds, content_preds_scaled)]
alpha_06_rmse = np.sqrt(mean_squared_error(actual, alpha_06_preds))

print("\n" + "="*70)
print("COMPREHENSIVE RESULTS")
print("="*70)

results = {
    'CF Only': cf_rmse,
    'Content Only (original)': 1.1208,  # From previous eval
    f'Content Only (scaled {best_scale}x)': content_rmse,
    'Hybrid (α=0.6, original content)': 0.9062,  # From previous eval
    f'Hybrid (α=0.6, scaled content)': alpha_06_rmse,
    'Hybrid (adaptive α, scaled content)': adaptive_rmse
}

print("\nMethod                                    | RMSE   | vs Baseline")
print("------------------------------------------|--------|------------")
baseline = cf_rmse
for method, rmse in results.items():
    improvement = ((baseline - rmse) / baseline) * 100
    symbol = "✓" if rmse < baseline else "✗"
    print(f"{method:41} | {rmse:.4f} | {symbol} {improvement:+.1f}%")

# Find best method
best_method = min(results.items(), key=lambda x: x[1])
print(f"\n🏆 Best Method: {best_method[0]}")
print(f"   RMSE: {best_method[1]:.4f}")
print(f"   Improvement over CF: {((cf_rmse - best_method[1]) / cf_rmse * 100):.1f}%")

# ============================================================
# ANALYZE BY USER ACTIVITY SEGMENTS
# ============================================================
print("\n" + "="*70)
print("PERFORMANCE BY USER ACTIVITY LEVEL")
print("="*70)

# Segment users by activity
test_sample['user_activity'] = test_sample['user_id'].map(user_activity)

segments = [
    ('Sparse (1-10 ratings)', 1, 10),
    ('Medium (11-30 ratings)', 11, 30),
    ('Active (31+ ratings)', 31, 1000)
]

print("\nSegment              | Size  | CF RMSE | Content | Hybrid α=0.6 | Adaptive")
print("---------------------|-------|---------|---------|--------------|----------")

for seg_name, min_act, max_act in segments:
    seg_mask = (test_sample['user_activity'] >= min_act) & (test_sample['user_activity'] <= max_act)
    seg_indices = test_sample[seg_mask].index.tolist()
    
    if len(seg_indices) == 0:
        continue
    
    # Get predictions for this segment
    seg_actual = [actual[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    seg_cf = [cf_preds[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    seg_cont = [content_preds_scaled[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    seg_alpha06 = [alpha_06_preds[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    seg_adapt = [adaptive_preds[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    
    rmse_cf = np.sqrt(mean_squared_error(seg_actual, seg_cf))
    rmse_cont = np.sqrt(mean_squared_error(seg_actual, seg_cont))
    rmse_a06 = np.sqrt(mean_squared_error(seg_actual, seg_alpha06))
    rmse_adapt = np.sqrt(mean_squared_error(seg_actual, seg_adapt))
    
    print(f"{seg_name:20} | {len(seg_indices):5} | {rmse_cf:.4f}  | {rmse_cont:.4f}  | {rmse_a06:.4f}       | {rmse_adapt:.4f}")

# ============================================================
# SAVE IMPROVED RESULTS
# ============================================================
improved_results = {
    'best_scale_factor': best_scale,
    'all_methods': results,
    'best_method': best_method[0],
    'best_rmse': best_method[1],
    'segment_analysis': segments,
    'predictions': {
        'cf': cf_preds,
        'content_scaled': content_preds_scaled,
        'hybrid_fixed': alpha_06_preds,
        'hybrid_adaptive': adaptive_preds,
        'actual': actual
    }
}

joblib.dump(improved_results, 'improved_validation_results.pkl')
print(f"\n✓ Results saved to 'improved_validation_results.pkl'")

print("\n" + "="*70)
print("EVALUATION COMPLETE!")
print("="*70)

print("\n📊 Key Takeaways:")
print("  1. Variance scaling improves content predictions")
print("  2. Adaptive alpha helps different user types")
print("  3. Segmented analysis shows where each method excels")
print("  4. Ready for Phase 4 comprehensive evaluation!")

IMPROVED HYBRID RECOMMENDER - EVALUATION

[1/4] Loading model and validation data...
✓ Loaded 10,000 validation samples

[2/4] Testing variance-scaled content predictions...

Testing content variance scaling factors:
Factor | Content RMSE | Hybrid RMSE (α=0.6)
-------|--------------|--------------------
  1.0  |    1.1208    |      0.9062
  1.5  |    1.1347    |      0.9088
  2.0  |    1.1466    |      0.9111
  2.5  |    1.1619    |      0.9142
  3.0  |    1.1687    |      0.9155

✓ Best scale factor: 1.0 (Hybrid RMSE: 0.9062)

[3/4] Testing confidence-weighted hybrid (adaptive alpha)...

Adaptive alpha strategy:
  User Activity | Alpha | Method Weighting
  --------------|-------|------------------
  0-5 ratings   | 0.30  | 30% CF + 70% Content (favor content)
  25 ratings    | 0.55  | 55% CF + 45% Content (balanced)
  50+ ratings   | 0.80  | 80% CF + 20% Content (favor CF)

✓ Adaptive hybrid RMSE: 0.9662

[4/4] Final comparison of all methods...

COMPREHENSIVE RESULTS

Method         